# Iniciando o Spark



In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
from pyspark.sql import SparkSession
import os
import pytz
from datetime import datetime

spark = SparkSession.builder.appName("tabelas").getOrCreate()

In [3]:
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

In [4]:
spark

# Instalando bibliotecas

In [5]:

import os
import sys
import pytz
import numpy as np
import datetime
from pyspark.sql import SparkSession
from pyspark.sql import SQLContext
from pyspark.sql.functions import udf,split, lpad, concat_ws,to_timestamp,col,coalesce
from datetime import datetime
from datetime import timedelta
from datetime import date
from dateutil.relativedelta import relativedelta
from pyspark.sql.types import *
from pyspark.sql.functions import count, avg, to_date
from pyspark.sql import functions as F

#Configuração do pipeline( contem configuração do spark e os paths para serem alterados)


In [6]:
# caminhos dos arquivos

PATH_TABELA_BUREAU ="/content/gdrive/MyDrive/Raw Hackathon PoD 2025/base_score_bureau_movel_full"
PATH_TABELA_CADASTRO = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/base_dados_cadastrais"
PATH_BASE_TELCO       = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/base_telco"
PATH_BASE_RECARGA     = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/bases_recarga/BI_FP_ASS_RECARGA_CMV_NOVA"
PATH_BASE_BOOK_ATRASO = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/book_atraso/dados_faturamento"
PATH_BASE_BOOK_PAGAMENTO ="/content/gdrive/MyDrive/Raw Hackathon PoD 2025/book_pagamento/dados_pagamento"

BASE_PATH_RECARGA = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/bases_recarga"

PATH_DIMENSOES_RECARGA = {
    "CANAL_AQUISICAO": f"{BASE_PATH_RECARGA}/BI_DIM_CANAL_AQUISICAO_CREDITO.csv",
    "FORMA_PAGAMENTO": f"{BASE_PATH_RECARGA}/BI_DIM_FORMA_PAGAMENTO.csv",
    "INSTITUICAO": f"{BASE_PATH_RECARGA}/BI_DIM_INSTITUICAO.csv",
    "PLATAFORMA": f"{BASE_PATH_RECARGA}/BI_DIM_PLATAFORMA.csv",
    "PROMOCAO": f"{BASE_PATH_RECARGA}/BI_DIM_PROMOCAO_CREDITO.csv",
    "STATUS_PLATAFORMA": f"{BASE_PATH_RECARGA}/BI_DIM_STATUS_PLATAFORMA.csv",
    "TECNOLOGIA": f"{BASE_PATH_RECARGA}/BI_DIM_TECNOLOGIA.csv",
    "TIPO_CREDITO": f"{BASE_PATH_RECARGA}/BI_DIM_TIPO_CREDITO.csv",
    "TIPO_INSERCAO": f"{BASE_PATH_RECARGA}/BI_DIM_TIPO_INSERCAO.csv",
    "TIPO_RECARGA": f"{BASE_PATH_RECARGA}/BI_DIM_TIPO_RECARGA.csv",
}


PATH_DIMENSOES_BOOK_ATRASO = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/book_atraso/BI_DIM_TIPO_FATURAMENTO.csv"

In [7]:
# Csvs da tabela recarga

BASE_PATH_RECARGA = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/bases_recarga"

DIMENSOES_RECARGA = {
    "CANAL_AQUISICAO_CREDITO": "BI_DIM_CANAL_AQUISICAO_CREDITO.csv",
    "FORMA_PAGAMENTO": "BI_DIM_FORMA_PAGAMENTO.csv",
    "INSTITUICAO": "BI_DIM_INSTITUICAO.csv",
    "PLANO_PRECO": "BI_DIM_PLANO_PRECO.csv",
    "PLATAFORMA": "BI_DIM_PLATAFORMA.csv",
    "PROMOCAO_CREDITO": "BI_DIM_PROMOCAO_CREDITO.csv",
    "STATUS_PLATAFORMA": "BI_DIM_STATUS_PLATAFORMA.csv",
    "TECNOLOGIA": "BI_DIM_TECNOLOGIA.csv",
    "TIPO_CREDITO": "BI_DIM_TIPO_CREDITO.csv",
    "TIPO_INSERCAO": "BI_DIM_TIPO_INSERCAO.csv",
    "TIPO_RECARGA": "BI_DIM_TIPO_RECARGA.csv",
}


dfs_dim_recarga = {}

for nome_dim, arquivo in DIMENSOES_RECARGA.items():
    path = f"{BASE_PATH_RECARGA}/{arquivo}"

    dfs_dim_recarga[nome_dim] = (
        spark.read
        .option("header", True)
        .option("sep", ",")
        .option("inferSchema", True)
        .csv(path)
    )

df_CANAL_AQUISICAO_CREDITO = dfs_dim_recarga["CANAL_AQUISICAO_CREDITO"]
df_FORMA_PAGAMENTO        = dfs_dim_recarga["FORMA_PAGAMENTO"]
df_INSTITUICAO            = dfs_dim_recarga["INSTITUICAO"]
df_PLANO_PRECO            = dfs_dim_recarga["PLANO_PRECO"]
df_PLATAFORMA             = dfs_dim_recarga["PLATAFORMA"]
df_PROMOCAO_CREDITO       = dfs_dim_recarga["PROMOCAO_CREDITO"]
df_STATUS_PLATAFORMA      = dfs_dim_recarga["STATUS_PLATAFORMA"]
df_TECNOLOGIA              = dfs_dim_recarga["TECNOLOGIA"]
df_TIPO_CREDITO           = dfs_dim_recarga["TIPO_CREDITO"]
df_TIPO_INSERCAO          = dfs_dim_recarga["TIPO_INSERCAO"]
df_TIPO_RECARGA           = dfs_dim_recarga["TIPO_RECARGA"]

In [8]:
# cadastro
df_cadastro = spark.read.parquet(PATH_TABELA_CADASTRO)
df_cadastro.createOrReplaceTempView("df_cadastro")

# bureau
df_bureau = spark.read.parquet(PATH_TABELA_BUREAU)
df_bureau.createOrReplaceTempView("df_bureau")

# telco
df_base_telco = spark.read.parquet(PATH_BASE_TELCO)
df_base_telco.createOrReplaceTempView("df_base_telco")

# recarga
df_recarga = spark.read.parquet(PATH_BASE_RECARGA)
df_recarga.createOrReplaceTempView("df_recarga")

# book atraso
df_book_atraso = spark.read.parquet(PATH_BASE_BOOK_ATRASO)
df_book_atraso.createOrReplaceTempView("df_book_atraso")

# book pagamento
df_book_pagamento = spark.read.parquet(PATH_BASE_BOOK_PAGAMENTO)
df_book_pagamento.createOrReplaceTempView("df_book_pagamento")


Estrutura Geral da Solução

O processamento foi organizado em camadas lógicas, utilizando CTEs (WITH) em SQL Spark para garantir clareza, performance e reprodutibilidade no ambiente do Google Colab.

Consolidação Mensal por Número (CPF + Safra + Telefone)

Nesta etapa, os dados são agregados ao nível de:

CPF

Safra (mês de referência)

Número de telefone (DW_NUM_CLIENTE)

Objetivo:

Identificar quanto cada número contribuiu em crédito no mês, permitindo ranking e análise de concentração.

Métrica gerada:

VAL_CREDITO_MES → valor total de crédito por número no mês

Quantidades de Telefones por Cpf




# Book de Pagamentos – Resumo

 Objetivo
Construir uma **tabela analítica de pagamentos e créditos**, agregada no nível **CPF + Safra**, considerando que um CPF pode possuir **mais de um número de telefone**.

 Fonte
Base transacional de pagamentos (`BOOK_PAGAMENTOS`), com informações de crédito, faturas e números de telefone.

 Estratégia
 Consolidação do crédito mensal no nível **CPF**
 Contagem de números distintos por CPF
 Criação de **flag indicativa de múltiplos números**
 Construção de **janelas temporais móveis** (U1M, U3M, U6M, U9M, U12M)
 Processamento realizado integralmente em **Spark SQL**

 Principais Variáveis
 `NUM_CPF` – Identificador do cliente  
 `SAFRA` – Mês de referência  
 `DW_NUM_CLIENTE` – Quantidade de números associados ao CPF  
 `FLG_UNICO_NUMERO` – 1 = único número | 0 = mais de um número  
 `VAL_CREDITO_MES_CPF` – Crédito total do CPF no mês  
 `U1M / U3M / U6M / U9M / U12M` – Acumulados de crédito por janela temporal  

 Resultado
Tabela final com **1 registro por CPF e Safra**, pronta para:
 Modelos preditivos
 Feature Store
 Consumo analítico (BI)



In [9]:
# book pagamento
df_book_pagamento = spark.read.parquet(PATH_BASE_BOOK_PAGAMENTO)
df_book_pagamento.createOrReplaceTempView("df_book_pagamento")

In [10]:
df_book_pagamento.createOrReplaceTempView("df_book_pagamento")

df_book_pagamento = spark.sql("""
SELECT
CAST(NUM_CPF AS STRING) AS NUM_CPF,
TO_DATE(DAT_STATUS_FATURA, 'ddMMMyyyy:HH:mm:ss') AS DAT_STATUS_FATURA,
CAST(CONTRATO AS STRING) AS CONTRATO,
CAST(SEQ_FATURA AS INT) AS SEQ_FATURA,
CAST(NUM_SUB_SEQ_FATURA AS INT) AS NUM_SUB_SEQ_FATURA,
CAST(NUM_CREDITO_SEQ AS INT) AS NUM_CREDITO_SEQ,
CAST(DW_TIPO_FATURA AS INT) AS DW_TIPO_FATURA,
CAST(IND_STATUS_FATURA AS STRING) AS IND_STATUS_FATURA,
CAST(DW_NUM_CLIENTE AS STRING) AS DW_NUM_CLIENTE,
CAST(DW_AREA AS INT) AS DW_AREA,
CAST(DW_UN_NEGOCIO AS INT) AS DW_UN_NEGOCIO,
CAST(DW_FORMA_PAGAMENTO AS INT) AS DW_FORMA_PAGAMENTO,
CAST(VAL_PAGAMENTO_FATURA AS FLOAT) AS VAL_PAGAMENTO_FATURA,
TO_DATE(DAT_CRIACAO_DW, 'ddMMMyyyy:HH:mm:ss') AS DAT_CRIACAO_DW,
TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_DW, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss') AS HR_CRIACAO_DW,
CAST(DW_BANCO AS STRING) AS DW_BANCO,
CAST(DW_TIPO_PAGAMENTO AS STRING) AS DW_TIPO_PAGAMENTO,
CAST(NUM_BANCO_PAGAMENTO AS STRING) AS NUM_BANCO_PAGAMENTO,
CAST(NUM_AGENCIA_PAGAMENTO AS STRING) AS NUM_AGENCIA_PAGAMENTO,
CAST(NUM_CC_PAGAMENTO AS STRING) AS NUM_CC_PAGAMENTO,
CAST(DW_MOTIVO_ESTORNO AS STRING) AS DW_MOTIVO_ESTORNO,
CAST(VAL_DESCONTO_ITEM AS FLOAT) AS VAL_DESCONTO_ITEM,
CAST(VAL_PAGAMENTO_ITEM AS FLOAT) AS VAL_PAGAMENTO_ITEM,
CAST(VAL_JUROS_MULTAS_ITEM AS FLOAT) AS VAL_JUROS_MULTAS_ITEM,
CAST(VAL_MULTA_EQUIP_ITEM AS FLOAT) AS VAL_MULTA_EQUIP_ITEM,
CAST(VAL_MULTA_EQUIP_TOTAL AS FLOAT) AS VAL_MULTA_EQUIP_TOTAL,
CAST(VAL_MULTA_FID_ITEM AS FLOAT) AS VAL_MULTA_FID_ITEM,
CAST(COD_ORIGEM_NETUNO AS STRING) AS COD_ORIGEM_NETUNO,
CAST(COD_CONTA_ATIVIDADE AS STRING) AS COD_CONTA_ATIVIDADE,
CAST(SEQ_ENTIDADE_ATIVIDADE AS INT) AS SEQ_ENTIDADE_ATIVIDADE,
TO_DATE(DAT_CRIACAO_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss')                              AS DAT_CRIACAO_ATIVIDADE,
TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')     AS HR_CRIACAO_ATIVIDADE,
TO_DATE(DAT_ATUALIZACAO_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss')                          AS DAT_ATUALIZACAO_ATIVIDADE,
TO_CHAR(TO_TIMESTAMP(DAT_ATUALIZACAO_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss') AS HR_ATUALIZACAO_ATIVIDADE,
CAST(COD_LOGIN_OPERADOR_ATIVIDADE AS STRING) AS COD_LOGIN_OPERADOR_ATIVIDADE,
CAST(COD_ATIVIDADE AS STRING) AS COD_ATIVIDADE,
CAST(COD_RAZAO_ATIVIDADE AS STRING) AS COD_RAZAO_ATIVIDADE,
TO_DATE(DAT_BAIXA_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss') AS DAT_BAIXA_ATIVIDADE,
CAST(VAL_BAIXA_ATIVIDADE AS FLOAT) AS VAL_BAIXA_ATIVIDADE,
TO_DATE(DAT_DEPOSITO_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss') AS DAT_DEPOSITO_ATIVIDADE,
CAST(COD_FUNDO_ATIVIDADE AS STRING) AS COD_FUNDO_ATIVIDADE,
CAST(COD_BANCO_ATIVIDADE AS STRING) AS COD_BANCO_ATIVIDADE,
CAST(NUM_CONTA_ATIVIDADE AS STRING) AS NUM_CONTA_ATIVIDADE,
CAST(COD_AGENCIA_ATIVIDADE AS STRING) AS COD_AGENCIA_ATIVIDADE,
CAST(SEQ_ENTIDADE_PAGAMENTO AS INT) AS SEQ_ENTIDADE_PAGAMENTO,
TO_DATE(DAT_CRIACAO_PAGAMENTO, 'ddMMMyyyy:HH:mm:ss')                              AS DAT_CRIACAO_PAGAMENTO,
TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_PAGAMENTO, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')     AS HR_CRIACAO_PAGAMENTO,
TO_DATE(DAT_ATUALIZACAO_PAGAMENTO, 'ddMMMyyyy:HH:mm:ss')                          AS DAT_ATUALIZACAO_PAGAMENTO,
TO_CHAR(TO_TIMESTAMP(DAT_ATUALIZACAO_PAGAMENTO, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss') AS HR_ATUALIZACAO_PAGAMENTO,
CAST(COD_LOGIN_PAGAMENTO AS STRING) AS COD_LOGIN_PAGAMENTO,
CAST(COD_FORMA_PAGAMENTO AS STRING) AS COD_FORMA_PAGAMENTO,
CAST(VAL_ORIGINAL_PAGAMENTO AS FLOAT) AS VAL_ORIGINAL_PAGAMENTO,
CAST(NUM_FATURA_PAGAMENTO AS STRING) AS NUM_FATURA_PAGAMENTO,
CAST(COD_TIPO_PAGAMENTO AS STRING) AS COD_TIPO_PAGAMENTO,
CAST(DSC_NOME_BANCO_PAGAMENTO AS STRING) AS DSC_NOME_BANCO_PAGAMENTO,
CAST(SEQ_ARQUIVO_PAGAMENTO AS INT) AS SEQ_ARQUIVO_PAGAMENTO,
CAST(NUM_PARCELA_PAGAMENTO AS INT) AS NUM_PARCELA_PAGAMENTO,
CAST(NUM_AGRUPADOR_PAGAMENTO AS INT) AS NUM_AGRUPADOR_PAGAMENTO,
CAST(DSC_PAGAMENTO AS STRING) AS DSC_PAGAMENTO,
CAST(VAL_ATUAL_PAGAMENTO AS FLOAT) AS VAL_ATUAL_PAGAMENTO,
CAST(COD_METODO_PAGAMENTO AS INT) AS COD_METODO_PAGAMENTO,
CAST(IND_STATUS_PAGAMENTO AS STRING) AS IND_STATUS_PAGAMENTO,
TO_DATE(DAT_STATUS_PAGAMENTO, 'ddMMMyyyy:HH:mm:ss') AS DAT_STATUS_PAGAMENTO,
CAST(COD_ARQUIVO_PAGAMENTO AS STRING) AS COD_ARQUIVO_PAGAMENTO,
CAST(COD_NETUNO_PAGAMENTO AS STRING) AS COD_NETUNO_PAGAMENTO,
TO_DATE(DAT_CRIACAO_CREDITO, 'ddMMMyyyy:HH:mm:ss')                          AS DAT_CRIACAO_CREDITO,
TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_CREDITO, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss') AS HR_CRIACAO_CREDITO,
TO_DATE(DAT_ATUALIZACAO_CREDITO, 'ddMMMyyyy:HH:mm:ss')                      AS DAT_ATUALIZACAO_CREDITO,
CAST(COD_LOGIN_CREDITO AS STRING) AS COD_LOGIN_CREDITO,
CAST(VAL_PAGAMENTO_CREDITO AS FLOAT) AS VAL_PAGAMENTO_CREDITO,
CAST(IND_TIPO_CREDITO AS STRING) AS IND_TIPO_CREDITO,
CAST(SEQ_PAGAMENTO_CREDITO AS INT) AS SEQ_PAGAMENTO_CREDITO,
CAST(SEQ_FATURA_CREDITO AS INT) AS SEQ_FATURA_CREDITO,
CAST(COD_ALOCACAO_CREDITO AS STRING) AS COD_ALOCACAO_CREDITO,
CAST(COD_DESALOCACAO_CREDITO AS STRING) AS COD_DESALOCACAO_CREDITO,
CAST(SEQ_ENTIDADE_CREDITO AS INT) AS SEQ_ENTIDADE_CREDITO,
CAST(COD_TIPO_FATURA AS STRING) AS COD_TIPO_FATURA,
TO_DATE(DAT_ATIVIDADE_CREDITO, 'ddMMMyyyy:HH:mm:ss') AS DAT_ATIVIDADE_CREDITO,
TO_DATE(DAT_VENCIMENTO_CREDITO, 'ddMMMyyyy:HH:mm:ss') AS DAT_VENCIMENTO_CREDITO

FROM df_book_pagamento
""")

In [11]:
#Criando coluna safra
df_book_pagamento.createOrReplaceTempView("df_book_pagamento")

df_book_pagamento_01 = spark.sql("""
SELECT
    *,
    TO_DATE(
        CONCAT(
            YEAR(DAT_STATUS_FATURA), '-',
            LPAD(MONTH(DAT_STATUS_FATURA), 2, '0'), '-01'
        )
    ) AS SAFRA
FROM df_book_pagamento
""")

# colocar indentificador nas variaveis da tabela book_pagamento
for c in df_book_pagamento_01.columns:
    df_book_pagamento_01 = df_book_pagamento_01.withColumnRenamed(
        c, f"BOOK_PGTO_{c}"
    )
df_book_pagamento = df_book_pagamento_01

In [31]:
df_book_pagamento_01.createOrReplaceTempView("df_book_pagamento_01")

df_book_3 = spark.sql("""

WITH base_numero_mes AS (
    SELECT
        BOOK_PGTO_NUM_CPF,
        BOOK_PGTO_SAFRA,
        BOOK_PGTO_DW_NUM_CLIENTE AS NUMERO,
        SUM(BOOK_PGTO_VAL_ORIGINAL_PAGAMENTO) AS VAL_PAGO_MES
    FROM df_book_pagamento_01
    GROUP BY
        BOOK_PGTO_NUM_CPF,
        BOOK_PGTO_SAFRA,
        BOOK_PGTO_DW_NUM_CLIENTE
),

credito_numero_janelas AS (
    SELECT
        *,
        SUM(VAL_PAGO_MES) OVER (
            PARTITION BY BOOK_PGTO_NUM_CPF, NUMERO
            ORDER BY BOOK_PGTO_SAFRA
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS VAL_CREDITO_U3M_NUMERO
    FROM base_numero_mes
),

numeros_agg AS (
    SELECT
        BOOK_PGTO_NUM_CPF,
        BOOK_PGTO_SAFRA,

        COUNT(DISTINCT NUMERO) AS QTD_NUMEROS_CPF,

        COUNT(DISTINCT CASE
            WHEN COALESCE(VAL_CREDITO_U3M_NUMERO, 0) = 0
            THEN NUMERO
        END) AS QTD_NUM_INATIVOS_1A3M,

        SUM(VAL_PAGO_MES) AS VAL_CREDITO_TOTAL_MES
    FROM credito_numero_janelas
    GROUP BY
        BOOK_PGTO_NUM_CPF,
        BOOK_PGTO_SAFRA
),

book_financeiro_mensal AS (
    SELECT
        BOOK_PGTO_NUM_CPF,
        BOOK_PGTO_SAFRA,

        COUNT(DISTINCT BOOK_PGTO_SEQ_FATURA) AS QTD_FATURAS_MES,
        SUM(BOOK_PGTO_VAL_PAGAMENTO_FATURA) AS VAL_PAGO_FATURA_MES,
        SUM(BOOK_PGTO_VAL_DESCONTO_ITEM) AS VAL_DESCONTO_MES,
        SUM(BOOK_PGTO_VAL_JUROS_MULTAS_ITEM) AS VAL_JUROS_MULTAS_MES,
        SUM(BOOK_PGTO_VAL_MULTA_EQUIP_TOTAL) AS VAL_MULTA_EQUIP_MES,
        SUM(BOOK_PGTO_VAL_ORIGINAL_PAGAMENTO) AS VAL_PAGO_MES
    FROM df_book_pagamento_01
    GROUP BY
        BOOK_PGTO_NUM_CPF,
        BOOK_PGTO_SAFRA
),

book_financeiro_janelas AS (
    SELECT
        *,
        VAL_PAGO_FATURA_MES AS VAL_PAGO_U1M,

        SUM(VAL_PAGO_FATURA_MES) OVER (
            PARTITION BY BOOK_PGTO_NUM_CPF
            ORDER BY BOOK_PGTO_SAFRA
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS VAL_PAGO_U3M,

        SUM(VAL_PAGO_FATURA_MES) OVER (
            PARTITION BY BOOK_PGTO_NUM_CPF
            ORDER BY BOOK_PGTO_SAFRA
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ) AS VAL_PAGO_U6M,

        SUM(VAL_PAGO_FATURA_MES) OVER (
            PARTITION BY BOOK_PGTO_NUM_CPF
            ORDER BY BOOK_PGTO_SAFRA
            ROWS BETWEEN 8 PRECEDING AND CURRENT ROW
        ) AS VAL_PAGO_U9M,

        SUM(VAL_PAGO_FATURA_MES) OVER (
            PARTITION BY BOOK_PGTO_NUM_CPF
            ORDER BY BOOK_PGTO_SAFRA
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS VAL_PAGO_U12M
    FROM book_financeiro_mensal
)

SELECT
    f.BOOK_PGTO_NUM_CPF,
    f.BOOK_PGTO_SAFRA,
    f.QTD_FATURAS_MES,

    CAST(f.VAL_PAGO_FATURA_MES  AS DECIMAL(18,2)) AS VAL_PAGO_FATURA_MES,
    CAST(f.VAL_DESCONTO_MES     AS DECIMAL(18,2)) AS VAL_DESCONTO_MES,
    CAST(f.VAL_JUROS_MULTAS_MES AS DECIMAL(18,2)) AS VAL_JUROS_MULTAS_MES,
    CAST(f.VAL_MULTA_EQUIP_MES  AS DECIMAL(18,2)) AS VAL_MULTA_EQUIP_MES,
    CAST(f.VAL_PAGO_MES      AS DECIMAL(18,2)) AS VAL_PAGO_MES,

    CAST(f.VAL_PAGO_U1M  AS DECIMAL(18,2)) AS VAL_PAGO_U1M,
    CAST(f.VAL_PAGO_U3M  AS DECIMAL(18,2)) AS VAL_PAGO_U3M,
    CAST(f.VAL_PAGO_U6M  AS DECIMAL(18,2)) AS VAL_PAGO_U6M,
    CAST(f.VAL_PAGO_U9M  AS DECIMAL(18,2)) AS VAL_PAGO_U9M,
    CAST(f.VAL_PAGO_U12M AS DECIMAL(18,2)) AS VAL_PAGO_U12M,

    CASE
        WHEN n.QTD_NUMEROS_CPF > 1 THEN 1
        ELSE 0
    END AS FLG_MAIS_DE_UM_NUMERO

FROM book_financeiro_janelas f
LEFT JOIN numeros_agg n
    ON f.BOOK_PGTO_NUM_CPF = n.BOOK_PGTO_NUM_CPF
   AND f.BOOK_PGTO_SAFRA   = n.BOOK_PGTO_SAFRA

""")


In [32]:
df_book_3.show()

+-----------------+---------------+---------------+-------------------+----------------+--------------------+-------------------+------------+------------+------------+------------+------------+-------------+---------------------+
|BOOK_PGTO_NUM_CPF|BOOK_PGTO_SAFRA|QTD_FATURAS_MES|VAL_PAGO_FATURA_MES|VAL_DESCONTO_MES|VAL_JUROS_MULTAS_MES|VAL_MULTA_EQUIP_MES|VAL_PAGO_MES|VAL_PAGO_U1M|VAL_PAGO_U3M|VAL_PAGO_U6M|VAL_PAGO_U9M|VAL_PAGO_U12M|FLG_MAIS_DE_UM_NUMERO|
+-----------------+---------------+---------------+-------------------+----------------+--------------------+-------------------+------------+------------+------------+------------+------------+-------------+---------------------+
|      777778UZTN8|     2024-07-01|              1|              54.90|            0.00|                0.00|               0.00|       54.90|       54.90|      109.70|      109.70|      109.70|       109.70|                    0|
|      777778UZTN8|     2024-09-01|              1|              54.90|     